# Soma máxima de subarray contíguo

**Problema.** Dado um vetor `A[0..n-1]` com números positivos e negativos, encontrar a maior soma possível de uma subsequência contígua.

Exemplo usado na atividade:

```python
A = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
```

A resposta esperada é `6`, produzida pelo subarray `[4, -1, 2, 1]`.

Este notebook implementa, em **Cython**, cinco versões do problema:

1. algoritmo cúbico ingênuo;
2. algoritmo quadrático evitando recomputação incremental;
3. algoritmo quadrático com soma prefixada;
4. algoritmo por divisão e conquista;
5. algoritmo linear de Kadane.

In [1]:
%load_ext cython

## Dados de teste

Usaremos um vetor pequeno para validação e vetores maiores para comparar tempo de execução. A convenção adotada é permitir o subarray vazio, portanto, se todos os valores forem negativos, a resposta será `0`.

In [2]:
A_exemplo = [31, -41, 59, 26, -53, 58, 97, -93, -23, 84]
A_negativo = [-8, -3, -10]
print(A_exemplo)

[31, -41, 59, 26, -53, 58, 97, -93, -23, 84]


## 1. Solução cúbica — força bruta completa

A solução ingênua testa todos os pares `(i, j)` e, para cada par, percorre novamente o intervalo `A[i..j]` para calcular a soma.

### Ideia

Para cada início `i`:

- escolher cada fim `j >= i`;
- somar todos os elementos entre `i` e `j`;
- atualizar a melhor soma encontrada.

### Análise assintótica detalhada

O algoritmo possui três laços aninhados:

- o laço de `i` executa `n` vezes;
- para cada `i`, o laço de `j` executa aproximadamente `n - i` vezes;
- para cada par `(i, j)`, o laço de `k` percorre o tamanho do subarray.

O número total de somas é proporcional a:

\[
\sum_{i=0}^{n-1}\sum_{j=i}^{n-1}(j-i+1)
= 
rac{n(n+1)(n+2)}{6}
\]

Logo, o tempo é **Θ(n³)**. A linha crítica é a soma interna `soma += A[k]`, pois ela recalcula do zero valores que já foram calculados para subarrays anteriores.

In [7]:
%%cython
# cython: boundscheck=False, wraparound=False, cdivision=True

def max_subarray_cubico(list A):
    cdef Py_ssize_t n = len(A)
    cdef Py_ssize_t i, j, k
    cdef long soma, melhor = 0

    for i in range(n):
        for j in range(i, n):
            soma = 0
            for k in range(i, j + 1):
                soma += <long>A[k]
            if soma > melhor:
                melhor = soma
    return melhor

In [17]:
print(max_subarray_cubico(A_exemplo))
print(max_subarray_cubico(A_negativo))

6
0


## 2. Solução quadrática incremental — removendo a recomputação direta

A primeira melhoria elimina o terceiro laço. Em vez de recalcular `A[i] + ... + A[j]` do zero para cada `j`, acumulamos a soma conforme o limite direito avança.

### Melhoria sobre o cúbico

No algoritmo cúbico, ao calcular `A[i..j]`, a soma de `A[i..j-1]` é perdida. Aqui, ela é preservada em `soma`.

### Análise assintótica

Agora há apenas dois laços relevantes:

\[
\sum_{i=0}^{n-1}(n-i) = 
rac{n(n+1)}{2}
\]

Cada iteração faz trabalho constante. Portanto, o tempo é **Θ(n²)** e o espaço extra é **Θ(1)**.

A otimização remove exatamente a linha crítica do algoritmo cúbico: o laço interno que somava novamente o subarray inteiro.

In [6]:
%%cython
# cython: boundscheck=False, wraparound=False, cdivision=True

def max_subarray_quadratico_incremental(list A):
    cdef Py_ssize_t n = len(A)
    cdef Py_ssize_t i, j
    cdef long soma, melhor = 0

    for i in range(n):
        soma = 0
        for j in range(i, n):
            soma += <long>A[j]
            if soma > melhor:
                melhor = soma
    return melhor

In [15]:
print(max_subarray_quadratico_incremental(A_exemplo))
print(max_subarray_quadratico_incremental(A_negativo))

6
0


## 3. Solução quadrática com soma prefixada

Outra forma de evitar recomputações é criar um vetor auxiliar `prefixo`, em que:

\[
prefixo[t] = A[0] + A[1] + \cdots + A[t-1]
\]

Assim, a soma de `A[i..j]` pode ser obtida em tempo constante:

\[
soma(i,j) = prefixo[j+1] - prefixo[i]
\]

### Análise assintótica

- Construção do prefixo: **Θ(n)**;
- enumeração de todos os pares `(i, j)`: **Θ(n²)**;
- cálculo de cada soma: **Θ(1)**.

Logo, o tempo total é **Θ(n²)** e o espaço adicional é **Θ(n)**.

Comparada à versão incremental, a complexidade de tempo é a mesma, mas a técnica é útil para consultas repetidas de soma de intervalo.

In [5]:
%%cython
# cython: boundscheck=False, wraparound=False, cdivision=True

def max_subarray_quadratico_prefixo(list A):
    cdef Py_ssize_t n = len(A)
    cdef Py_ssize_t i, j
    cdef list prefixo = [0] * (n + 1)
    cdef long soma, melhor = 0

    for i in range(n):
        prefixo[i + 1] = <long>prefixo[i] + <long>A[i]

    for i in range(n):
        for j in range(i, n):
            soma = <long>prefixo[j + 1] - <long>prefixo[i]
            if soma > melhor:
                melhor = soma
    return melhor

In [13]:
print(max_subarray_quadratico_prefixo(A_exemplo))
print(max_subarray_quadratico_prefixo(A_negativo))

6
0


## 4. Divisão e conquista

A ideia é dividir o vetor em duas metades. A melhor soma pode estar:

1. totalmente na metade esquerda;
2. totalmente na metade direita;
3. cruzando o meio.

O caso cruzando o meio é calculado procurando:

- a melhor soma que termina no meio, olhando para a esquerda;
- a melhor soma que começa após o meio, olhando para a direita.

### Recorrência

Em cada chamada, resolvemos dois subproblemas de tamanho `n/2` e fazemos uma varredura linear para combinar:

\[
T(n) = 2T(n/2) + \Theta(n)
\]

Pelo Teorema Mestre, o tempo é **Θ(n log n)**. O espaço da pilha recursiva é **Θ(log n)**.

In [3]:
%%cython
# cython: boundscheck=False, wraparound=False, cdivision=True

cdef long _max_subarray_dc(list A, Py_ssize_t esquerda, Py_ssize_t direita):
    cdef Py_ssize_t meio, i
    cdef long soma, melhor_esq, melhor_dir
    cdef long max_esq, max_dir, max_cruzando

    if esquerda > direita:
        return 0
    if esquerda == direita:
        if <long>A[esquerda] > 0:
            return <long>A[esquerda]
        return 0

    meio = (esquerda + direita) // 2

    soma = 0
    melhor_esq = 0
    for i in range(meio, esquerda - 1, -1):
        soma += <long>A[i]
        if soma > melhor_esq:
            melhor_esq = soma

    soma = 0
    melhor_dir = 0
    for i in range(meio + 1, direita + 1):
        soma += <long>A[i]
        if soma > melhor_dir:
            melhor_dir = soma

    max_cruzando = melhor_esq + melhor_dir
    max_esq = _max_subarray_dc(A, esquerda, meio)
    max_dir = _max_subarray_dc(A, meio + 1, direita)

    if max_esq >= max_dir and max_esq >= max_cruzando:
        return max_esq
    if max_dir >= max_esq and max_dir >= max_cruzando:
        return max_dir
    return max_cruzando

def max_subarray_divisao_conquista(list A):
    return _max_subarray_dc(A, 0, len(A) - 1)

In [4]:
print(max_subarray_divisao_conquista(A_exemplo))
print(max_subarray_divisao_conquista(A_negativo))

187
0


## 5. Algoritmo linear — Kadane

O algoritmo linear mantém duas informações:

- `max_terminando_aqui`: melhor soma de um subarray que termina exatamente na posição atual;
- `melhor`: melhor soma encontrada em qualquer posição até agora.

Se `max_terminando_aqui + A[i]` ficar negativo, descartamos o prefixo e reiniciamos em zero, pois a convenção permite o subarray vazio.

### Invariante

Após processar `A[0..i]`:

- `max_terminando_aqui` contém a melhor soma de um subarray que termina em `i`;
- `melhor` contém a melhor soma de qualquer subarray dentro de `A[0..i]`.

### Análise assintótica

O vetor é percorrido uma única vez, e cada elemento executa trabalho constante. Logo:

- tempo: **Θ(n)**;
- espaço adicional: **Θ(1)**.

Este é assintoticamente ótimo, pois qualquer algoritmo precisa olhar todos os `n` elementos ao menos uma vez.

In [8]:
%%cython
# cython: boundscheck=False, wraparound=False, cdivision=True

def max_subarray_kadane(list A):
    cdef Py_ssize_t i, n = len(A)
    cdef long max_terminando_aqui = 0
    cdef long melhor = 0

    for i in range(n):
        max_terminando_aqui += <long>A[i]
        if max_terminando_aqui < 0:
            max_terminando_aqui = 0
        if max_terminando_aqui > melhor:
            melhor = max_terminando_aqui
    return melhor

In [11]:
print(max_subarray_kadane(A_exemplo))
print(max_subarray_kadane(A_negativo))

187
0


## Validação conjunta

Todas as versões devem retornar o mesmo resultado para o vetor de exemplo.

In [9]:
funcoes = [
    max_subarray_cubico,
    max_subarray_quadratico_incremental,
    max_subarray_quadratico_prefixo,
    max_subarray_divisao_conquista,
    max_subarray_kadane,
]

for f in funcoes:
    print(f.__name__, f(A_exemplo))

max_subarray_cubico 187
max_subarray_quadratico_incremental 187
max_subarray_quadratico_prefixo 187
max_subarray_divisao_conquista 187
max_subarray_kadane 187


## Comparação experimental simples

Evite usar tamanhos muito grandes no algoritmo cúbico, pois ele cresce rapidamente. A comparação abaixo usa tamanhos pequenos para manter o notebook executável em sala.

In [10]:
import random
import time

random.seed(42)

for n in [20, 50, 100, 200, 500]:
    A = [random.randint(-20, 20) for _ in range(n)]

    print(f"\nn = {n}")

    for f in funcoes:
        ini = time.perf_counter()
        r = f(A)
        fim = time.perf_counter()

        print(f"{f.__name__:40s} resultado={r:5d} tempo={fim-ini:.6f}s")


n = 20
max_subarray_cubico                      resultado=   30 tempo=0.000016s
max_subarray_quadratico_incremental      resultado=   30 tempo=0.000005s
max_subarray_quadratico_prefixo          resultado=   30 tempo=0.000009s
max_subarray_divisao_conquista           resultado=   30 tempo=0.000006s
max_subarray_kadane                      resultado=   30 tempo=0.000003s

n = 50
max_subarray_cubico                      resultado=   71 tempo=0.000206s
max_subarray_quadratico_incremental      resultado=   71 tempo=0.000007s
max_subarray_quadratico_prefixo          resultado=   71 tempo=0.000013s
max_subarray_divisao_conquista           resultado=   71 tempo=0.000007s
max_subarray_kadane                      resultado=   71 tempo=0.000001s

n = 100
max_subarray_cubico                      resultado=  133 tempo=0.001095s
max_subarray_quadratico_incremental      resultado=  133 tempo=0.000047s
max_subarray_quadratico_prefixo          resultado=  133 tempo=0.000059s
max_subarray_divisao_conqu

## Síntese das complexidades

| Algoritmo | Técnica principal | Tempo | Espaço extra |
|---|---:|---:|---:|
| Cúbico | força bruta com soma interna | Θ(n³) | Θ(1) |
| Quadrático incremental | salvar soma parcial | Θ(n²) | Θ(1) |
| Quadrático com prefixo | pré-processamento de somas | Θ(n²) | Θ(n) |
| Divisão e conquista | dividir, resolver e combinar | Θ(n log n) | Θ(log n) |
| Kadane | programação dinâmica escaneando | Θ(n) | Θ(1) |